# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

---

### Research Question

**Can we predict which content pages are underperforming their expected click-through rate (CTR) based on their search position tier, and rank them into a priority queue so content teams know which pages to rewrite first?**

### Decision Supported

This work supports a content editor's decision of **which pages to rewrite first** to recover lost search clicks. Given a portfolio of thousands of pages across dozens of client websites, editors cannot manually review every page — they need a ranked queue that surfaces the highest-impact opportunities at the top.

### Who Acts and What They Do

A content editor or SEO strategist reads the ranked queue top-down, opens the highest-ranked pages, and rewrites the title tag, meta description, or on-page content to improve click-through rate. Each page review takes approximately 1–2 hours of editorial effort.

### Cost of a Wrong Call

- **False positive** (flagging a well-performing page): ~1–2 hours of editor time wasted per page reviewed. No permanent damage.
- **False negative** (missing a high-impression, low-CTR page): the page continues losing clicks every day it sits at a good position with a poor snippet — opportunity cost compounds over weeks.

This asymmetry means **Precision@K at small K** (top 20–50) is the right metric: the team has limited review capacity, so every queue slot must count.

### Why Data/ML Helps

CTR expectations differ by position tier — a 0.2% CTR at positions 1–3 is alarming, but the same CTR at position 30+ is expected. A flat CTR threshold would generate false positives at deep positions and miss real problems at top positions. A position-adjusted scoring system (whether rule-based or learned) accounts for position, volume, engagement, and content depth simultaneously — the pattern is real but too multi-dimensional for a single manual threshold.

In [ ]:
# == Cell 1: Connect to warehouse + verify data availability ==
import duckdb
import os, sys
import pandas as pd
import numpy as np
import pathlib, getpass

# Load HF_TOKEN from .env
_env = pathlib.Path(os.getcwd()).resolve()
for _ in range(5):
    _ep = _env / '.env'
    if _ep.exists():
        for _line in _ep.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _env = _env.parent

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}
print('DuckDB connected.')

# Quick verification: row counts and date range
for name, tbl in TABLES.items():
    row_count = con.sql(f'SELECT COUNT(*) AS n FROM {tbl}').fetchone()[0]
    print(f'  {name}: {row_count:,} rows')

date_range = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_sample']}
""").fetchone()
print(f'  Date range: {date_range[0]} to {date_range[1]}')

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

---

### Data Sources

1. **Starter CSV** (`data/raw/content_refresh_anonymized.csv`) — 30,000 rows × 44 columns, 32 pseudonymized clients. Used in `w01_research_question.ipynb` for initial exploration and lane selection only. Not used in any modeling or evaluation.

2. **Warehouse dataset** (Hugging Face, gated) — `hf://datasets/FlyRank/internship-warehouse`, build v20260703. Used for all modeling work (w03–w05):

| Table | Rows | Purpose |
|---|---|---|
| `fact_content_daily_performance_sample` | ~11.7M | Daily search + analytics metrics, aggregated to monthly grain |
| `dim_content` | 519,606 | Page properties (`word_count`) joined via `content_hash_id` |
| `dim_clients` | 104 | Client profiles for holdout split validation |

### Time Window

**June 2026** (`month = '2026-06'`) — the latest complete calendar month in the sample dataset. After filtering for `impressions >= 500` and `avg_position > 0`, this yields **52,766 pages** across 44 clients.

### Deliberately Excluded

**Label leakage (would inflate scores dishonestly):**

| Excluded Field | Why |
|---|---|
| `observed_ctr` | Directly encodes the target — `is_opportunity` is defined by comparing observed CTR to tier median. |
| `ctr_gap` | IS the label definition (`ctr_gap > 0` → opportunity). Used only in label construction, never as a model feature. |
| `trend_direction` | Post-hoc product flag derived from `trend_pct`. Trailing outcome that leaks future information. |
| `trend_pct` | Label-derived field from the starter pipeline. Excluded to prevent leakage. |

**ID fields (grouping only, never features):**

| Excluded Field | Why |
|---|---|
| `client_hash_id` | Pseudonymized client ID. Used for grouped train/test splits only. |
| `content_hash_id` | Pseudonymized page ID. Used for joins and row identification only. |

**Privacy:** All identifiers are pseudonymized hashes. No real client names, website URLs, or search queries appear anywhere in `work/`.

In [ ]:
# == Cell 2: Build feature vector for June 2026 ==

feature_vector_q = f"""
WITH monthly_agg_raw AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)        AS total_impressions,
        SUM(f.gsc_clicks)             AS total_clicks,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0
             ELSE 0.0
        END AS observed_ctr,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions)
             ELSE 0.0
        END AS avg_position,
        CASE WHEN SUM(f.ga4_sessions) > 0
             THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0
             ELSE 0.0
        END AS engagement_rate,
        MAX(CASE WHEN f.ga4_data_available = TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        ANY_VALUE(d.word_count) AS word_count
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-06'
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING total_impressions >= 500 AND avg_position > 0
),
monthly_agg AS (
    SELECT m.*,
        CASE
            WHEN m.avg_position <= 3  THEN 'pos_1_3'
            WHEN m.avg_position <= 10 THEN 'pos_4_10'
            WHEN m.avg_position <= 20 THEN 'pos_11_20'
            WHEN m.avg_position <= 50 THEN 'pos_21_50'
            ELSE 'pos_51_plus'
        END AS position_tier
    FROM monthly_agg_raw m
)
SELECT m.*, t.tier_median_ctr
FROM monthly_agg m
LEFT JOIN (
    SELECT position_tier, MEDIAN(observed_ctr) AS tier_median_ctr
    FROM monthly_agg
    GROUP BY position_tier
) t ON m.position_tier = t.position_tier
"""

df = con.sql(feature_vector_q).df()
df['word_count'] = df['word_count'].fillna(0)
df['log_impressions'] = np.log1p(df['total_impressions'])
df['ctr_gap'] = df['tier_median_ctr'] - df['observed_ctr']
df['is_opportunity'] = ((df['ctr_gap'] > 0) & (df['total_impressions'] >= 1000)).astype(int)

print(f'Total pages (June 2026): {len(df):,}')
print(f'Unique clients: {df["client_hash_id"].nunique()}')
print(f'Label: is_opportunity = 1 where ctr_gap > 0 AND impressions >= 1,000')
print(f'Base rate (is_opportunity): {df["is_opportunity"].mean():.4f} ({df["is_opportunity"].mean()*100:.1f}%)')
print(f'\nFeature vector shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

---

### Label Definition

A page is labeled `is_opportunity = 1` if **both** conditions hold:
1. Its observed CTR is **below** its position tier's median CTR (`ctr_gap > 0`)
2. It receives at least **1,000 impressions** per month (enough traffic to justify editorial effort)

The base rate is **33.7%** — roughly one-third of visible pages are underperforming.

### Honest Feature Vector (5 features)

From our leakage audit (`w03_feature_leakage_check.ipynb`), we use only features that do NOT leak into the label:

| Feature | What it measures |
|---|---|
| `log_impressions` | log1p(total search views) — traffic volume, log-scaled |
| `avg_position` | Impression-weighted average SERP rank |
| `engagement_rate` | GA4 engaged sessions / total sessions × 100 |
| `word_count` | Page length in words (0 if missing) |
| `has_ga4_data` | Whether GA4 tracking is active (0/1 flag) |

**Banned:** `observed_ctr` (leaks into label), `ctr_gap` (IS the label definition), `trend_direction`/`trend_pct` (trailing outcomes).

### Baseline: Hand-Coded Rule

A transparent, deterministic scoring rule built in `w04_baseline_score.ipynb`:

```
baseline_score = visible × clickable × underperforming × ctr_gap × log_impressions
```

Where:
- `visible` = 1 if impressions >= 1,000
- `clickable` = 1 if position <= 20
- `underperforming` = 1 if ctr_gap > 0
- The score scales with the size of the gap and traffic volume

This baseline achieves **100% Precision@K** across all K values because it directly uses `ctr_gap` — the metric that defines the label. It is a strong, fair reference point.

### Validation Design

**GroupShuffleSplit by `client_hash_id`** — 80% train / 20% test, `random_state=42`.

Entire clients go into either train OR test, never both. This prevents client-level memorization and tests whether models generalize to **unseen clients** — the real deployment scenario.

### Model Selection

Following the "readable → stronger" ladder:
1. **Logistic Regression** — transparent, readable coefficients, establishes a learned baseline
2. **Random Forest** (200 trees, max_depth=8) — handles non-linear relationships without manual feature engineering

Both are compared against the rule baseline on the **same test split** using **Precision@K** and **ROC AUC**.

### Leakage Checks

Performed in `w03_feature_leakage_check.ipynb`: each candidate feature was tested by computing its AUC against the label. `observed_ctr` and `ctr_gap` showed near-perfect AUC and were permanently banned. The final 5-feature set shows no suspiciously perfect signal.

In [ ]:
# == Cell 3: Baseline rule + ML models (LR + RF) on grouped client split ==
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)

# ---- Feature matrix ----
FEATURES = ['log_impressions', 'avg_position', 'engagement_rate', 'word_count', 'has_ga4_data']
X = df[FEATURES].values
y = df['is_opportunity'].values
groups = df['client_hash_id'].values

# ---- Grouped client split ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Split: Train {len(train_idx):,} pages ({len(set(groups[train_idx]))} clients) | '
      f'Test {len(test_idx):,} pages ({len(set(groups[test_idx]))} clients)')
print(f'Train label rate: {y_train.mean():.4f} | Test label rate: {y_test.mean():.4f}')

# ---- Rule baseline on test set ----
df_test = df.iloc[test_idx].copy()
visible = (df_test['total_impressions'] >= 1000).astype(int)
clickable = (df_test['avg_position'] <= 20).astype(int)
underperforming = (df_test['ctr_gap'] > 0).astype(int)
baseline_scores = (visible * clickable * underperforming * df_test['ctr_gap'] * df_test['log_impressions']).values

# ---- Logistic Regression ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]

# ---- Random Forest ----
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

print(f'\nModels trained. LR coefficients: {dict(zip(FEATURES, lr.coef_[0].round(3))))}')
print(f'RF top feature (Gini): {FEATURES[np.argmax(rf.feature_importances_)]} ({rf.feature_importances_.max():.3f})')

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.